<a href="https://colab.research.google.com/github/luke-wellerman/luke-wellerman-ds2002-fa26/blob/main/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_mask = cat['product_name'].str.lower().str.contains('pop', na=False)
pop_tarts = cat[pop_mask]
print(len(pop_tarts))
pop_tarts

3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
# TODO
straw_mask = pop_tarts['product_name'].str.lower().str.contains('straw', na=False)
strawberry_pop_tarts = pop_tarts[straw_mask]
other_pop = pop_tarts[straw_mask == False]
print(strawberry_pop_tarts[['sku', 'product_name']])
print('\nOther:')
print(other_pop[['sku', 'product_name']])

             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

Other:
Empty DataFrame
Columns: [sku, product_name]
Index: []


Pop-Tart rows that are a different product: There are none in this data but they will be kept seperate and listed in the other data frame when present.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# TODO
mapping = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}
cat['cleanSku'] = cat['sku'].map(mapping).fillna(cat['sku'])
cat

,sku,product_name,category,cleanSku
0,PT-12,pop-tarts strawberry,Breakfast Foods,PT-12
1,SKU#459812,Strawberry Pop-Tart,Breakfast,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
# TODO
print('Before:')
print(cat['category'].value_counts(dropna=False))
cat['category'] = (cat['category']
                   .str.strip()
                   .str.title()
                   .replace({'Breakfast Foods': 'Breakfast'}))
print('\nAfter:')
print(cat['category'].value_counts(dropna=False))

Before:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

After:
category
Breakfast    3
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# TODO
cat['normalized'] = (cat['product_name'].astype(str).str.lower()
                     .str.replace(r'[^a-z0-9 ]', '', regex=True)
                     .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])
candidates

,sku
normalized,
poptarts strawberry,"[PT-12, POPTART-STRAW]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['cleanSku'].notna().all()
assert len(cat['cleanSku']) == len(cat)

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A) Water bottles appear fragmented, as there are 2 24pks that are from the same brand, while there is another 24pk from a seperate brand.  To fix this, the same items (water bottles) should be considered one sku but each brand should be kept seperate. For flashlights, there are multiple SKUs for one product. These should all be consolidated together.

B) Consolidation becomes an issue when considering products with the same item but different quantity. To know whether to consilidate product types or keep each product seperated by quantity, it depends on what the data is needed for. I would ask the person who requested the analysis. If looking for a specific product, it may be best to consolidate all variations of one item.